In [2]:
"""
=============================================================================
Edge-Fog Based Predictive Maintenance System
NASA CMAPSS Turbofan Engine Dataset - ML Classification Model Training
-----------------------------------------------------------------------------
GOOGLE COLAB + GOOGLE DRIVE VERSION
=============================================================================
INSTRUCTIONS:
  1. Upload this file to Google Colab  (or paste into a notebook cell)
  2. Run Cell 0 to mount your Google Drive
  3. Set DRIVE_DATA_PATH to the folder on your Drive that contains:
         train_FD001.txt  (or FD002 / FD003 / FD004)
  4. Run all cells — model + 10 graphs will be saved back to Drive
=============================================================================
Dataset  : FD001 (change DATASET_ID below for FD002/FD003/FD004)
Model    : Random Forest Classifier
Target   : Binary classification
           0 → Healthy   (RUL > threshold)
           1 → At Risk   (RUL ≤ threshold)
=============================================================================
"""

'\n=============================================================================\nEdge-Fog Based Predictive Maintenance System\nNASA CMAPSS Turbofan Engine Dataset - ML Classification Model Training\n-----------------------------------------------------------------------------\nGOOGLE COLAB + GOOGLE DRIVE VERSION\n=============================================================================\nINSTRUCTIONS:\n  1. Upload this file to Google Colab  (or paste into a notebook cell)\n  2. Run Cell 0 to mount your Google Drive\n  3. Set DRIVE_DATA_PATH to the folder on your Drive that contains:\n         train_FD001.txt  (or FD002 / FD003 / FD004)\n  4. Run all cells — model + 10 graphs will be saved back to Drive\n=============================================================================\nDataset  : FD001 (change DATASET_ID below for FD002/FD003/FD004)\nModel    : Random Forest Classifier\nTarget   : Binary classification\n           0 → Healthy   (RUL > threshold)\n           1 → At Risk 

In [3]:
import os, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, learning_curve
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)
from matplotlib.patches import Patch
import joblib

warnings.filterwarnings("ignore")

In [4]:
DRIVE_DATA_PATH = "/content/drive/MyDrive/Edge-Fog-based-Predictive-Maintenance-System-for-Manufacturing-Equipment-main/dataset/CMaps"

# ── Output folder (graphs + model will be saved here) ────────────────────────
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/Fog-Outputs"

# ── Dataset to use: "FD001" | "FD002" | "FD003" | "FD004" ───────────────────
DATASET_ID = "FD001"

# ── Classification threshold: cycles ≤ this → "At Risk" ─────────────────────
RUL_THRESHOLD = 30

# ── Piece-wise linear RUL cap ─────────────────────────────────────────────────
MAX_RUL_CLIP = 125

# ── Model hyper-parameters ───────────────────────────────────────────────────
N_ESTIMATORS = 200
RANDOM_STATE = 42
TEST_SIZE    = 0.20
CV_FOLDS     = 5

# ─────────────────────────────────────────────────────────────────────────────
# (nothing below this line needs to be changed)
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs(DRIVE_OUTPUT_PATH, exist_ok=True)

SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]
OP_COLS     = ["op_setting_1", "op_setting_2", "op_setting_3"]
COL_NAMES   = (["unit_id", "cycle"] + OP_COLS + SENSOR_COLS
               + ["sensor_22", "sensor_23"])

# Sensors with near-zero variance in FD001
DROP_SENSORS = ["sensor_1","sensor_5","sensor_6",
                "sensor_10","sensor_16","sensor_18","sensor_19"]

In [5]:
def load_cmapss(dataset_id: str) -> pd.DataFrame:
    path = os.path.join(DRIVE_DATA_PATH, f"train_{dataset_id}.txt")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\n❌  File not found: {path}"
            f"\n   Please update DRIVE_DATA_PATH to the correct folder on your Drive."
        )
    df = pd.read_csv(path, sep=r"\s+", header=None, names=COL_NAMES)
    df.drop(columns=["sensor_22", "sensor_23"], inplace=True)

    max_cycles = df.groupby("unit_id")["cycle"].max().rename("max_cycle")
    df         = df.merge(max_cycles, on="unit_id")
    df["RUL"]  = df["max_cycle"] - df["cycle"]
    df.drop(columns=["max_cycle"], inplace=True)

    df["RUL"]   = df["RUL"].clip(upper=MAX_RUL_CLIP)
    df["label"] = (df["RUL"] <= RUL_THRESHOLD).astype(int)
    return df


def build_features(df: pd.DataFrame):
    keep_sensors = [s for s in SENSOR_COLS if s not in DROP_SENSORS]
    grp = df.groupby("unit_id")
    for col in keep_sensors:
        df[f"{col}_rmean"] = grp[col].transform(
            lambda x: x.rolling(5, min_periods=1).mean())
        df[f"{col}_rstd"]  = grp[col].transform(
            lambda x: x.rolling(5, min_periods=1).std().fillna(0))

    base_features = (keep_sensors + OP_COLS
                     + [f"{c}_rmean" for c in keep_sensors]
                     + [f"{c}_rstd"  for c in keep_sensors])
    return df[base_features].copy(), df["label"].copy()


def save_fig(name: str):
    path = os.path.join(DRIVE_OUTPUT_PATH, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()          # display inline in Colab
    plt.close()
    print(f"   ✅  Saved → {path}")


PALETTE = {"Healthy (0)": "#2196F3", "At Risk (1)": "#F44336"}

# ─────────────────────────────────────────────────────────────────────────────
print("=" * 62)
print("  PREDICTIVE MAINTENANCE — CLASSIFICATION MODEL TRAINING")
print("=" * 62)
print(f"\n[1/9] Loading CMAPSS {DATASET_ID} from Drive …")
df = load_cmapss(DATASET_ID)
print(f"      Rows : {len(df):,}  |  Engines : {df['unit_id'].nunique()}")
print(f"      Label distribution →\n{df['label'].value_counts().to_string()}")

  PREDICTIVE MAINTENANCE — CLASSIFICATION MODEL TRAINING

[1/9] Loading CMAPSS FD001 from Drive …
      Rows : 20,631  |  Engines : 100
      Label distribution →
label
0    17531
1     3100


In [6]:
print("\n[2/9] Generating EDA plots …")

# ── 01. Class Distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Class Distribution — Failure Risk Labels",
             fontsize=14, fontweight="bold")
counts = df["label"].value_counts().rename({0: "Healthy (0)", 1: "At Risk (1)"})
axes[0].bar(counts.index, counts.values,
            color=[PALETTE["Healthy (0)"], PALETTE["At Risk (1)"]],
            edgecolor="white", linewidth=1.5, width=0.5)
axes[0].set_title("Count per Class"); axes[0].set_ylabel("Number of Cycles")
for i, (lbl, val) in enumerate(counts.items()):
    axes[0].text(i, val + 200,
                 f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=10)
axes[1].pie(counts.values, labels=counts.index,
            colors=[PALETTE["Healthy (0)"], PALETTE["At Risk (1)"]],
            autopct="%1.1f%%", startangle=140,
            wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("Class Proportion")
plt.tight_layout(); save_fig("01_class_distribution.png")

# ── 02. RUL Distribution ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors_arr = np.where(df["RUL"] <= RUL_THRESHOLD, "#F44336", "#2196F3")
ax.scatter(range(len(df)), df["RUL"].values, c=colors_arr, s=0.5, alpha=0.4)
ax.axhline(RUL_THRESHOLD, color="orange", linewidth=2,
           linestyle="--", label=f"Threshold = {RUL_THRESHOLD} cycles")
handles = [Patch(color="#2196F3", label="Healthy"),
           Patch(color="#F44336", label="At Risk"),
           plt.Line2D([0],[0], color="orange", linestyle="--",
                      label=f"Threshold = {RUL_THRESHOLD}")]
ax.legend(handles=handles, fontsize=10)
ax.set_title("RUL Distribution Across All Cycles",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Sample Index"); ax.set_ylabel("Remaining Useful Life (cycles)")
plt.tight_layout(); save_fig("02_rul_distribution.png")

# ── 03. Sensor Time-Series for Engine #1 ─────────────────────────────────────
keep_sensors   = [s for s in SENSOR_COLS if s not in DROP_SENSORS]
sample_engine  = df[df["unit_id"] == 1].sort_values("cycle")
n_sens = min(9, len(keep_sensors)); ncols = 3
nrows  = (n_sens + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()
fig.suptitle("Sensor Readings vs Cycle — Engine #1",
             fontsize=14, fontweight="bold")
for i, sensor in enumerate(keep_sensors[:n_sens]):
    c_arr = np.where(sample_engine["label"] == 1, "#F44336", "#2196F3")
    axes[i].scatter(sample_engine["cycle"], sample_engine[sensor],
                    c=c_arr, s=10, alpha=0.7)
    axes[i].set_title(sensor, fontsize=10)
    axes[i].set_xlabel("Cycle"); axes[i].set_ylabel("Value")
    axes[i].grid(alpha=0.3)
for j in range(n_sens, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout(); save_fig("03_sensor_timeseries.png")

# ── 04. Correlation Heatmap ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 11))
corr_df = df[keep_sensors + ["RUL", "label"]].corr()
mask    = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, annot_kws={"size": 7},
            cbar_kws={"shrink": 0.8})
ax.set_title("Sensor Feature Correlation Heatmap",
             fontsize=14, fontweight="bold")
plt.tight_layout(); save_fig("04_correlation_heatmap.png")

# ── 05. Per-Engine Degradation Curves ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
for uid in df["unit_id"].unique()[:20]:
    eng = df[df["unit_id"] == uid].sort_values("cycle")
    ax.plot(eng["cycle"], eng["RUL"], alpha=0.5, linewidth=1.2)
ax.axhline(RUL_THRESHOLD, color="red", linewidth=2, linestyle="--",
           label=f"Failure Threshold = {RUL_THRESHOLD}")
ax.set_title("RUL Degradation Curves — First 20 Engines",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Cycle"); ax.set_ylabel("RUL")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); save_fig("05_degradation_curves.png")

# ── 06. Sensor Boxplots by Label ─────────────────────────────────────────────
plot_sensors = keep_sensors[:6]
fig, axes    = plt.subplots(2, 3, figsize=(15, 8)); axes = axes.flatten()
fig.suptitle("Sensor Distribution: Healthy vs At Risk",
             fontsize=13, fontweight="bold")
for i, sensor in enumerate(plot_sensors):
    n_risk = (df["label"] == 1).sum()
    d0 = df[df["label"] == 0][sensor].sample(3000, random_state=42)
    d1 = df[df["label"] == 1][sensor].sample(min(3000, n_risk), random_state=42)
    axes[i].boxplot([d0, d1], labels=["Healthy", "At Risk"],
                    patch_artist=True,
                    boxprops=dict(facecolor="#90CAF9"),
                    medianprops=dict(color="navy", linewidth=2))
    axes[i].set_title(sensor, fontsize=10)
    axes[i].set_ylabel("Sensor Value"); axes[i].grid(alpha=0.3, axis="y")
plt.tight_layout(); save_fig("06_sensor_boxplots.png")

print("   EDA plots (01–06) complete.\n")


[2/9] Generating EDA plots …
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/01_class_distribution.png
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/02_rul_distribution.png
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/03_sensor_timeseries.png
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/04_correlation_heatmap.png
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/05_degradation_curves.png
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/06_sensor_boxplots.png
   EDA plots (01–06) complete.



In [7]:
print("[3/9] Feature engineering …")
X, y = build_features(df)
print(f"      Feature matrix : {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f"      Train : {X_train_sc.shape}  |  Test : {X_test_sc.shape}")

[3/9] Feature engineering …
      Feature matrix : (20631, 45)
      Train : (16504, 45)  |  Test : (4127, 45)


In [8]:
print("\n[4/9] Training Random Forest …")
clf = RandomForestClassifier(
    n_estimators      = N_ESTIMATORS,
    max_depth         = 20,
    min_samples_split = 4,
    min_samples_leaf  = 2,
    class_weight      = "balanced",
    n_jobs            = -1,
    random_state      = RANDOM_STATE
)
clf.fit(X_train_sc, y_train)
print("      Training complete ✅")


[4/9] Training Random Forest …
      Training complete ✅


In [9]:
print("\n[5/9] Evaluating model …")
y_pred  = clf.predict(X_test_sc)
y_proba = clf.predict_proba(X_test_sc)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
print(f"\n   Accuracy  : {acc*100:.2f}%")
print(f"   F1 Score  : {f1*100:.2f}%")
print(f"\n   Classification Report:\n")
print(classification_report(y_test, y_pred,
      target_names=["Healthy (0)", "At Risk (1)"]))


[5/9] Evaluating model …

   Accuracy  : 97.00%
   F1 Score  : 89.95%

   Classification Report:

              precision    recall  f1-score   support

 Healthy (0)       0.98      0.98      0.98      3507
 At Risk (1)       0.90      0.90      0.90       620

    accuracy                           0.97      4127
   macro avg       0.94      0.94      0.94      4127
weighted avg       0.97      0.97      0.97      4127



In [10]:
print("[6/9] Confusion matrix …")
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Confusion Matrix", fontsize=14, fontweight="bold")

ConfusionMatrixDisplay(cm, display_labels=["Healthy","At Risk"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Counts", fontsize=12)
for t in axes[0].texts: t.set_fontsize(16)

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=["Healthy","At Risk"]).plot(
    ax=axes[1], colorbar=False, cmap="Oranges", values_format=".2%")
axes[1].set_title("Normalized (Row %)", fontsize=12)
for t in axes[1].texts: t.set_fontsize(14)

plt.tight_layout(); save_fig("07_confusion_matrix.png")

[6/9] Confusion matrix …
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/07_confusion_matrix.png


In [11]:
print("[7/9] ROC & Precision-Recall curves …")
fpr, tpr, _  = roc_curve(y_test, y_proba)
roc_auc      = auc(fpr, tpr)
prec, rec, _ = precision_recall_curve(y_test, y_proba)
ap           = average_precision_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Model Performance Curves", fontsize=14, fontweight="bold")

axes[0].plot(fpr, tpr, color="#E91E63", lw=2.5,
             label=f"ROC  AUC = {roc_auc:.4f}")
axes[0].fill_between(fpr, tpr, alpha=0.1, color="#E91E63")
axes[0].plot([0,1],[0,1], "k--", lw=1.5, label="Random Classifier")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve"); axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, color="#4CAF50", lw=2.5, label=f"AP = {ap:.4f}")
axes[1].fill_between(rec, prec, alpha=0.1, color="#4CAF50")
axes[1].axhline(y_test.mean(), color="gray", linestyle="--",
                label=f"Baseline = {y_test.mean():.2f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.tight_layout(); save_fig("08_roc_pr_curves.png")
print(f"   ROC-AUC = {roc_auc:.4f}  |  Avg Precision = {ap:.4f}")

[7/9] ROC & Precision-Recall curves …
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/08_roc_pr_curves.png
   ROC-AUC = 0.9944  |  Avg Precision = 0.9721


In [12]:
print("\n[8/9] Feature importance …")
feat_imp = pd.Series(clf.feature_importances_, index=X.columns)
top20    = feat_imp.nlargest(20)

fig, ax = plt.subplots(figsize=(12, 7))
colors  = ["#F44336" if ("_rmean" in f or "_rstd" in f) else "#2196F3"
           for f in top20.index]
ax.barh(range(len(top20)), top20.values[::-1],
        color=colors[::-1], edgecolor="white", height=0.7)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=9)
ax.set_xlabel("Feature Importance (Gini)", fontsize=11)
ax.set_title("Top 20 Feature Importances — Random Forest",
             fontsize=13, fontweight="bold")
ax.grid(alpha=0.3, axis="x")
ax.legend(handles=[Patch(color="#F44336", label="Rolling stat features"),
                   Patch(color="#2196F3", label="Raw sensor features")],
          fontsize=9)
plt.tight_layout(); save_fig("09_feature_importance.png")


[8/9] Feature importance …
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/09_feature_importance.png


In [13]:
print("\n[9/9] Learning curves (may take ~1 min) …")
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
train_sizes, train_scores, val_scores = learning_curve(
    clf, X_train_sc, y_train,
    train_sizes=np.linspace(0.10, 1.0, 8),
    cv=cv, scoring="f1", n_jobs=-1, verbose=0
)
tr_m, tr_s = train_scores.mean(axis=1), train_scores.std(axis=1)
va_m, va_s = val_scores.mean(axis=1),   val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color="#2196F3")
ax.fill_between(train_sizes, va_m-va_s, va_m+va_s, alpha=0.15, color="#FF9800")
ax.plot(train_sizes, tr_m, "o-", color="#2196F3", lw=2.5, label="Training F1")
ax.plot(train_sizes, va_m, "s-", color="#FF9800", lw=2.5, label="Validation F1")
ax.set_title("Learning Curve (F1 Score)", fontsize=13, fontweight="bold")
ax.set_xlabel("Training Set Size"); ax.set_ylabel("F1 Score")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
plt.tight_layout(); save_fig("10_learning_curve.png")
print(f"   CV Val F1: {va_m[-1]*100:.2f}% ± {va_s[-1]*100:.2f}%")


[9/9] Learning curves (may take ~1 min) …
   ✅  Saved → /content/drive/MyDrive/Fog-Outputs/10_learning_curve.png
   CV Val F1: 88.76% ± 0.37%


In [14]:
model_path  = os.path.join(DRIVE_OUTPUT_PATH, "predictive_model_rf.pkl")
scaler_path = os.path.join(DRIVE_OUTPUT_PATH, "scaler.pkl")
joblib.dump(clf,    model_path)
joblib.dump(scaler, scaler_path)
print(f"\n   Model  saved → {model_path}")
print(f"   Scaler saved → {scaler_path}")


   Model  saved → /content/drive/MyDrive/Fog-Outputs/predictive_model_rf.pkl
   Scaler saved → /content/drive/MyDrive/Fog-Outputs/scaler.pkl


In [15]:
print("\n" + "=" * 62)
print("  FINAL RESULTS SUMMARY")
print("=" * 62)
print(f"  Dataset        : CMAPSS {DATASET_ID}")
print(f"  Total Samples  : {len(df):,}")
print(f"  Features Used  : {X.shape[1]}")
print(f"  RUL Threshold  : {RUL_THRESHOLD} cycles")
print(f"  Accuracy       : {acc*100:.2f}%")
print(f"  F1 Score       : {f1*100:.2f}%")
print(f"  ROC-AUC        : {roc_auc:.4f}")
print(f"  Avg Precision  : {ap:.4f}")
print(f"  CV Val F1      : {va_m[-1]*100:.2f}% ± {va_s[-1]*100:.2f}%")
print("=" * 62)
print(f"\n  All outputs saved to: {DRIVE_OUTPUT_PATH}")
print("""
  Graphs:
    01_class_distribution.png
    02_rul_distribution.png
    03_sensor_timeseries.png
    04_correlation_heatmap.png
    05_degradation_curves.png
    06_sensor_boxplots.png
    07_confusion_matrix.png
    08_roc_pr_curves.png
    09_feature_importance.png
    10_learning_curve.png

  Model files:
    predictive_model_rf.pkl
    scaler.pkl
""")


  FINAL RESULTS SUMMARY
  Dataset        : CMAPSS FD001
  Total Samples  : 20,631
  Features Used  : 45
  RUL Threshold  : 30 cycles
  Accuracy       : 97.00%
  F1 Score       : 89.95%
  ROC-AUC        : 0.9944
  Avg Precision  : 0.9721
  CV Val F1      : 88.76% ± 0.37%

  All outputs saved to: /content/drive/MyDrive/Fog-Outputs

  Graphs:
    01_class_distribution.png
    02_rul_distribution.png
    03_sensor_timeseries.png
    04_correlation_heatmap.png
    05_degradation_curves.png
    06_sensor_boxplots.png
    07_confusion_matrix.png
    08_roc_pr_curves.png
    09_feature_importance.png
    10_learning_curve.png
 
  Model files:
    predictive_model_rf.pkl
    scaler.pkl

